# Taxi Zones Bronze Quality Check

## Purpose

This notebook validates the existing Bronze Taxi Zones reference table:

`ftw-week-08`.`01-bronze`.taxi_zones_raw

The validation covers:

- Completeness
- Uniqueness
- Valid ranges
- Accepted categorical values
- Duplicate business records
- Special-location consistency
- EWR consistency
- Bronze ingestion metadata
- Source-to-Bronze reconciliation
- DQ result status and exit-gate conditions

### Important

This notebook is for **data quality validation only**.

It does not create, replace, clean, transform, or delete the Bronze
Taxi Zone table.

## Validation Approach

Each DQ check produces a standardized result.

### Status

| Status | Meaning |
|---|---|
| PASS | Validation expectation is satisfied |
| WARN | A non-critical or known source issue was detected and requires review |
| FAIL | A critical validation expectation was broken |
| INFO | Measurement only; does not affect the DQ pass rate |

### Severity

| Severity | Meaning |
|---|---|
| FAIL | Critical issue that should block downstream processing |
| WARN | Non-blocking issue requiring review |
| INFO | Informational measurement |

Checks are evaluated independently. A single record may contribute to more than
one check, so fail counts must not be summed to determine the number of unique
bad records.

## Dataset and Source Context

### Source

`taxi_zone_lookup.csv`

### Bronze Target

`ftw-week-08`.`01-bronze`.taxi_zones_raw

### Business Columns

- `location_id`
- `borough`
- `zone`
- `service_zone`

### Source Characteristics

The Taxi Zone Lookup is a reference dataset containing 265 records in the
current source snapshot.

`location_id` is the business identifier and should be unique and positive.

The source contains special classifications such as:

- `EWR`
- `Unknown`
- `Outside of NYC`

These are handled through source-specific consistency checks rather than being
automatically treated as data-quality failures.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS `ftw-week-08`.`02-bronze`.90_validate_taxi_zones (
    run_id STRING,
    executed_at TIMESTAMP,

    layer STRING,
    dataset STRING,

    batch_id STRING,
    source_version_id STRING,
    code_revision STRING,

    check_name STRING,
    check_type STRING,

    status STRING,
    severity STRING,

    fail_count BIGINT,
    total_count BIGINT,
    fail_pct DOUBLE,
    threshold_pct DOUBLE,
    metric_value DOUBLE,

    owner STRING,
    details STRING,
    evidence_location STRING
)
USING DELTA;

In [0]:
%sql
DECLARE OR REPLACE VARIABLE dq_run_id STRING;
SET VARIABLE dq_run_id = uuid();
SELECT
    dq_run_id AS run_id,
    current_timestamp() AS executed_at;

## Bronze Profile

The following query profiles the existing Bronze table before the formal DQ
checks are evaluated.

The profile is observational only and does not modify the Bronze table.

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT location_id) AS distinct_location_ids,

    SUM(CASE WHEN location_id IS NULL THEN 1 ELSE 0 END) AS null_location_id,
    SUM(CASE WHEN borough IS NULL THEN 1 ELSE 0 END) AS null_borough,
    SUM(CASE WHEN zone IS NULL THEN 1 ELSE 0 END) AS null_zone,
    SUM(CASE WHEN service_zone IS NULL THEN 1 ELSE 0 END) AS null_service_zone,

    MIN(location_id) AS min_location_id,
    MAX(location_id) AS max_location_id,

    SUM(CASE WHEN source_file IS NULL THEN 1 ELSE 0 END) AS null_source_file,
    SUM(CASE WHEN source_file_version IS NULL THEN 1 ELSE 0 END) AS null_source_version,
    SUM(CASE WHEN batch_id IS NULL THEN 1 ELSE 0 END) AS null_batch_id,
    SUM(CASE WHEN ingested_at IS NULL THEN 1 ELSE 0 END) AS null_ingested_at

FROM `ftw-week-08`.`02-bronze`.taxi_zones_raw;

## Taxi Zone DQ Checks

The checks below validate:

### Business Data

- Table is not empty
- `location_id` is not null
- `location_id` is unique
- `location_id` is positive
- Borough completeness
- Zone completeness
- Service-zone completeness
- Borough domain
- Service-zone domain
- Duplicate business records
- Special-location consistency
- EWR consistency

### Bronze Metadata

- Source file is present
- Source version is present
- Batch ID is present
- Ingestion timestamp is present

Known source-specific characteristics are reported as warnings where
appropriate instead of automatically blocking the pipeline.

In [0]:
%sql
WITH base AS (
    SELECT
        location_id,
        borough,
        zone,
        service_zone,
        source_file,
        source_file_version,
        batch_id,
        ingested_at
    FROM `ftw-week-08`.`02-bronze`.taxi_zones_raw
),
total AS (
    SELECT COUNT(*) AS total_count
    FROM base
),
checks AS (
    -- 1. Table is not empty
    SELECT
        'row_count_not_empty' AS check_name,
        'VOLUME' AS check_type,
        CASE WHEN total_count = 0 THEN 1 ELSE 0 END AS fail_count,
        1 AS total_count,
        0.0 AS threshold_pct,
        CAST(total_count AS DOUBLE) AS metric_value,
        'FAIL' AS severity,
        'Bronze Taxi Zone table must contain at least one record.' AS details
    FROM total
    UNION ALL
    -- 2. Location ID not null
    SELECT
        'location_id_not_null',
        'NOT_NULL',
        SUM(CASE WHEN location_id IS NULL THEN 1 ELSE 0 END),
        COUNT(*),
        0.0,
        NULL,
        'FAIL',
        'Every Taxi Zone record must have a location_id.'
    FROM base
    UNION ALL
    -- 3. Location ID unique
    SELECT
        'location_id_unique',
        'UNIQUE',
        COALESCE(
            (
                SELECT SUM(cnt - 1)
                FROM (
                    SELECT
                        location_id,
                        COUNT(*) AS cnt
                    FROM base
                    WHERE location_id IS NOT NULL
                    GROUP BY location_id
                    HAVING COUNT(*) > 1
                )
            ),
            0
        ),
        COUNT(*),
        0.0,
        NULL,
        'FAIL',
        'Non-null location_id values must be unique.'
    FROM base
    UNION ALL
    -- 4. Location ID positive
    SELECT
        'location_id_positive',
        'RANGE',
        SUM(
            CASE
                WHEN location_id IS NOT NULL
                     AND location_id <= 0
                THEN 1
                ELSE 0
            END
        ),
        COUNT(*),
        0.0,
        NULL,
        'FAIL',
        'location_id must be a positive identifier.'
    FROM base
    UNION ALL
    -- 5. Borough completeness
    SELECT
        'borough_not_null',
        'NOT_NULL',
        SUM(CASE WHEN borough IS NULL THEN 1 ELSE 0 END),
        COUNT(*),
        0.0,
        NULL,
        'WARN',
        'Null borough values require review.'
    FROM base
    UNION ALL
    -- 6. Zone completeness
    SELECT
        'zone_not_null',
        'NOT_NULL',
        SUM(CASE WHEN zone IS NULL THEN 1 ELSE 0 END),
        COUNT(*),
        0.0,
        NULL,
        'WARN',
        'Null zone values require review.'
    FROM base
    UNION ALL
    -- 7. Service zone completeness
    SELECT
        'service_zone_not_null',
        'NOT_NULL',
        SUM(CASE WHEN service_zone IS NULL THEN 1 ELSE 0 END),
        COUNT(*),
        0.0,
        NULL,
        'WARN',
        'Null service_zone values require review.'
    FROM base
    UNION ALL
    -- 8. Borough domain
    SELECT
        'borough_domain',
        'DOMAIN',
        SUM(
            CASE
                WHEN borough IS NOT NULL
                 AND TRIM(borough) NOT IN (
                    'EWR',
                    'Queens',
                    'Bronx',
                    'Manhattan',
                    'Staten Island',
                    'Brooklyn',
                    'Unknown'
                 )
                THEN 1
                ELSE 0
            END
        ),
        COUNT(*),
        0.0,
        NULL,
        'WARN',
        'Borough must use an accepted Taxi Zone source classification.'
    FROM base
    UNION ALL
    -- 9. Service zone domain
    SELECT
        'service_zone_domain',
        'DOMAIN',
        SUM(
            CASE
                WHEN service_zone IS NOT NULL
                 AND TRIM(service_zone) NOT IN (
                    'EWR',
                    'Boro Zone',
                    'Yellow Zone',
                    'Airports'
                 )
                THEN 1
                ELSE 0
            END
        ),
        COUNT(*),
        0.0,
        NULL,
        'WARN',
        'service_zone must use an accepted Taxi Zone source classification.'
    FROM base
    UNION ALL
    -- 10. Duplicate business records
    SELECT
        'full_row_unique',
        'UNIQUE',
        COALESCE(
            (
                SELECT SUM(cnt - 1)
                FROM (
                    SELECT
                        location_id,
                        borough,
                        zone,
                        service_zone,
                        COUNT(*) AS cnt
                    FROM base
                    GROUP BY
                        location_id,
                        borough,
                        zone,
                        service_zone
                    HAVING COUNT(*) > 1
                )
            ),
            0
        ),
        COUNT(*),
        0.0,
        NULL,
        'WARN',
        'Duplicate business-content rows should be reviewed.'
    FROM base
    UNION ALL
    -- 11. Special-location consistency
    SELECT
        'special_location_consistency',
        'CONSISTENCY',
        SUM(
            CASE
                WHEN UPPER(TRIM(borough)) = 'UNKNOWN'
                     AND (
                         zone IS NOT NULL
                         OR service_zone IS NOT NULL
                     )
                THEN 1

                WHEN UPPER(TRIM(zone)) = 'OUTSIDE OF NYC'
                     AND (
                         borough IS NOT NULL
                         OR service_zone IS NOT NULL
                     )
                THEN 1

                ELSE 0
            END
        ),
        COUNT(*),
        0.0,
        NULL,
        'WARN',
        'Special Taxi Zone classifications should remain internally consistent.'
    FROM base
    UNION ALL
    -- 12. EWR consistency
    SELECT
        'ewr_consistency',
        'CONSISTENCY',
        SUM(
            CASE
                WHEN UPPER(TRIM(borough)) = 'EWR'
                     AND (
                         COALESCE(UPPER(TRIM(service_zone)), '') <> 'EWR'
                         OR COALESCE(UPPER(TRIM(zone)), '') <> 'NEWARK AIRPORT'
                     )
                THEN 1

                WHEN UPPER(TRIM(service_zone)) = 'EWR'
                     AND (
                         COALESCE(UPPER(TRIM(borough)), '') <> 'EWR'
                         OR COALESCE(UPPER(TRIM(zone)), '') <> 'NEWARK AIRPORT'
                     )
                THEN 1

                ELSE 0
            END
        ),
        COUNT(*),
        0.0,
        NULL,
        'WARN',
        'EWR records should consistently map to Newark Airport and EWR.'
    FROM base
    UNION ALL
    -- 13. Source file metadata
    SELECT
        'source_file_not_null',
        'NOT_NULL',
        SUM(CASE WHEN source_file IS NULL THEN 1 ELSE 0 END),
        COUNT(*),
        0.0,
        NULL,
        'FAIL',
        'Every Bronze record must retain source file metadata.'
    FROM base
    UNION ALL
    -- 14. Source version metadata
    SELECT
        'source_version_not_null',
        'NOT_NULL',
        SUM(CASE WHEN source_file_version IS NULL THEN 1 ELSE 0 END),
        COUNT(*),
        0.0,
        NULL,
        'FAIL',
        'Every Bronze record must retain source version metadata.'
    FROM base
    UNION ALL
    -- 15. Batch ID metadata
    SELECT
        'batch_id_not_null',
        'NOT_NULL',
        SUM(CASE WHEN batch_id IS NULL THEN 1 ELSE 0 END),
        COUNT(*),
        0.0,
        NULL,
        'FAIL',
        'Every Bronze record must retain batch metadata.'
    FROM base
    UNION ALL
    -- 16. Ingestion timestamp metadata
    SELECT
        'ingested_at_not_null',
        'NOT_NULL',
        SUM(CASE WHEN ingested_at IS NULL THEN 1 ELSE 0 END),
        COUNT(*),
        0.0,
        NULL,
        'FAIL',
        'Every Bronze record must have an ingestion timestamp.'
    FROM base
),
measurements AS (
    -- 17. Row count measurement
    SELECT
        'row_count_measurement' AS check_name,
        'MEASURE' AS check_type,
        0 AS fail_count,
        COUNT(*) AS total_count,
        0.0 AS threshold_pct,
        CAST(COUNT(*) AS DOUBLE) AS metric_value,
        'INFO' AS severity,
        'Current Bronze Taxi Zone row count.' AS details
    FROM base
    UNION ALL
    -- 18. Location ID range measurement
    SELECT
        'location_id_range_measurement',
        'MEASURE',
        0,
        COUNT(*),
        0.0,
        CAST(MAX(location_id) - MIN(location_id) + 1 AS DOUBLE),
        'INFO',
        CONCAT(
            'LocationID range: ',
            CAST(MIN(location_id) AS STRING),
            ' to ',
            CAST(MAX(location_id) AS STRING)
        )
    FROM base
    UNION ALL
    -- 19. Location ID coverage measurement
    SELECT
        'location_id_coverage_measurement',
        'MEASURE',
        0,
        COUNT(*),
        0.0,
        CAST(COUNT(DISTINCT location_id) AS DOUBLE),
        'INFO',
        'Number of distinct LocationIDs currently present.'
    FROM base
),
all_checks AS (
    SELECT * FROM checks
    UNION ALL
    SELECT * FROM measurements
)
INSERT INTO `ftw-week-08`.`02-bronze`.90_validate_taxi_zones
SELECT
    dq_run_id AS run_id,
    current_timestamp() AS executed_at,
    'BRONZE' AS layer,
    'taxi_zones_raw' AS dataset,
    MAX(b.batch_id) AS batch_id,
    MAX(b.source_file_version) AS source_version_id,
    NULL AS code_revision,
    c.check_name,
    c.check_type,
    CASE
        WHEN c.severity = 'INFO' THEN 'INFO'
        WHEN c.fail_count = 0 THEN 'PASS'
        WHEN c.total_count IS NULL OR c.total_count = 0 THEN c.severity
        WHEN c.fail_count / NULLIF(c.total_count, 0) <= c.threshold_pct
            THEN 'WARN'
        ELSE c.severity
    END AS status,
    c.severity,
    CAST(c.fail_count AS BIGINT),
    CAST(c.total_count AS BIGINT),
    CASE
        WHEN c.total_count IS NULL OR c.total_count = 0 THEN NULL
        ELSE ROUND(c.fail_count / c.total_count * 100, 2)
    END AS fail_pct,
    c.threshold_pct,
    c.metric_value,
    'Data Engineering Team' AS owner,
    c.details,
    NULL AS evidence_location
FROM all_checks c
CROSS JOIN (
    SELECT
        MAX(batch_id) AS batch_id,
        MAX(source_file_version) AS source_file_version
    FROM base
) b
GROUP BY
    c.check_name,
    c.check_type,
    c.severity,
    c.fail_count,
    c.total_count,
    c.threshold_pct,
    c.metric_value,
    c.details;

## Review Current DQ Run

The following query displays all validation results generated for the current
run.

INFO measurements are retained for observability but are excluded from the
pass/fail evaluation.

In [0]:
%sql
SELECT
    check_name,
    check_type,
    status,
    severity,
    fail_count,
    total_count,
    fail_pct,
    threshold_pct,
    metric_value,
    details
FROM `ftw-week-08`.`02-bronze`.90_validate_taxi_zones
WHERE run_id = dq_run_id
ORDER BY
    CASE status
        WHEN 'FAIL' THEN 1
        WHEN 'WARN' THEN 2
        WHEN 'PASS' THEN 3
        WHEN 'INFO' THEN 4
    END,
    check_name;

## DQ Summary

The summary provides the number of checks in each status and the overall
pass rate.

INFO measurements are excluded from the pass-rate calculation.

In [0]:
%sql
SELECT
    COUNT(CASE WHEN status = 'PASS' THEN 1 END) AS passed_checks,
    COUNT(CASE WHEN status = 'WARN' THEN 1 END) AS warning_checks,
    COUNT(CASE WHEN status = 'FAIL' THEN 1 END) AS failed_checks,
    COUNT(CASE WHEN status = 'INFO' THEN 1 END) AS informational_checks,
    COUNT(
        CASE
            WHEN status IN ('PASS', 'WARN', 'FAIL')
            THEN 1
        END
    ) AS evaluated_checks,
    ROUND(
        100.0 * COUNT(CASE WHEN status = 'PASS' THEN 1 END)
        /
        NULLIF(
            COUNT(
                CASE
                    WHEN status IN ('PASS', 'WARN', 'FAIL')
                    THEN 1
                END
            ),
            0
        ),
        2
    ) AS pass_rate_pct
FROM `ftw-week-08`.`02-bronze`.90_validate_taxi_zones
WHERE run_id = dq_run_id;

## Bronze DQ Exit Gate

### Gate Rule

- Any `FAIL` result → Bronze validation is **BLOCKED**
- `WARN` results → validation may continue, but warnings must be reviewed
- `INFO` results → informational only

The gate is a validation decision and does not modify the Bronze table.

In [0]:
%sql
SELECT
    CASE
        WHEN COUNT(CASE WHEN status = 'FAIL' THEN 1 END) > 0
            THEN 'BLOCKED'
        ELSE 'PASSED'
    END AS bronze_dq_gate,

    COUNT(CASE WHEN status = 'FAIL' THEN 1 END) AS fail_count,
    COUNT(CASE WHEN status = 'WARN' THEN 1 END) AS warn_count,
    COUNT(CASE WHEN status = 'PASS' THEN 1 END) AS pass_count,
    COUNT(CASE WHEN status = 'INFO' THEN 1 END) AS info_count
FROM `ftw-week-08`.`02-bronze`.90_validate_taxi_zones
WHERE run_id = dq_run_id;

## Source → Bronze Reconciliation

Taxi Zone Lookup is a reference snapshot. The Bronze table should represent
the source snapshot without unintended loss, duplication, or modification.

The reconciliation checks:

1. Source row count vs Bronze row count
2. Source distinct LocationIDs vs Bronze distinct LocationIDs
3. Source business records missing from Bronze
4. Bronze business records not present in the source

The comparison uses the business columns:

- `location_id`
- `borough`
- `zone`
- `service_zone`

This is a validation-only operation.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW taxi_zone_source_validation AS
SELECT
    CAST(LocationID AS INT) AS location_id,
    Borough AS borough,
    Zone AS zone,
    service_zone
FROM read_files(
    '/Volumes/ftw-week-08/00-source/group_a_source/taxi_zones/taxi_zone_lookup.csv',
    format => 'csv',
    header => true
);

In [0]:
%sql
SELECT
    source_count,
    bronze_count,
    source_distinct_location_ids,
    bronze_distinct_location_ids,
    source_count - bronze_count AS row_count_difference,
    source_distinct_location_ids
        - bronze_distinct_location_ids AS distinct_id_difference
FROM (
    SELECT
        (
            SELECT COUNT(*)
            FROM taxi_zone_source_validation
        ) AS source_count,
        (
            SELECT COUNT(*)
            FROM `ftw-week-08`.`02-bronze`.taxi_zones_raw
        ) AS bronze_count,
        (
            SELECT COUNT(DISTINCT location_id)
            FROM taxi_zone_source_validation
        ) AS source_distinct_location_ids,
        (
            SELECT COUNT(DISTINCT location_id)
            FROM `ftw-week-08`.`02-bronze`.taxi_zones_raw
        ) AS bronze_distinct_location_ids
);

## Business-Content Reconciliation

Row counts alone are not sufficient to prove that the Bronze table matches the
source.

Two-way anti-joins are used to identify:

- Records present in the source but missing from Bronze
- Records present in Bronze but absent from the source

Expected result for an unchanged full-refresh snapshot:

**0 rows in both directions.**

In [0]:
%sql
SELECT
    s.*
FROM taxi_zone_source_validation s
LEFT ANTI JOIN `ftw-week-08`.`02-bronze`.taxi_zones_raw b
    ON s.location_id <=> b.location_id
   AND s.borough <=> b.borough
   AND s.zone <=> b.zone
   AND s.service_zone <=> b.service_zone
ORDER BY s.location_id;

In [0]:
%sql
SELECT
    b.location_id,
    b.borough,
    b.zone,
    b.service_zone
FROM `ftw-week-08`.`02-bronze`.taxi_zones_raw b
LEFT ANTI JOIN taxi_zone_source_validation s
    ON b.location_id <=> s.location_id
   AND b.borough <=> s.borough
   AND b.zone <=> s.zone
   AND b.service_zone <=> s.service_zone
ORDER BY b.location_id;

## WARN / FAIL Investigation

A non-zero DQ result should be investigated at record level before being
classified as a true defect.

This is especially important for the Taxi Zone source because it contains
special classifications such as:

- `Unknown`
- `Outside of NYC`
- `EWR`

The following query exposes those records for review.

In [0]:
%sql
SELECT
    location_id,
    borough,
    zone,
    service_zone
FROM `ftw-week-08`.`02-bronze`.taxi_zones_raw
WHERE
       UPPER(TRIM(borough)) = 'UNKNOWN'
    OR UPPER(TRIM(zone)) = 'OUTSIDE OF NYC'
    OR UPPER(TRIM(borough)) = 'EWR'
    OR UPPER(TRIM(service_zone)) = 'EWR'
ORDER BY location_id;

In [0]:
%sql
SELECT
    check_name,
    check_type,
    status,
    severity,
    fail_count,
    total_count,
    fail_pct,
    details
FROM `ftw-week-08`.`02-bronze`.90_validate_taxi_zones
WHERE run_id = dq_run_id
  AND status IN ('FAIL', 'WARN')
ORDER BY
    CASE status
        WHEN 'FAIL' THEN 1
        WHEN 'WARN' THEN 2
    END,
    check_name;